# ChEMBL Stratified Sampling

Generates a **quality-stratified** 100-molecule benchmark set.

**Strategy:**
1. Sample `N_OVERSAMPLE` molecules from `chembl_full.csv`
2. Featurize all of them
3. Call PrexSyn on all → compute actual `baseline_quality` for each
4. Bin by quality (<0.5, 0.5-0.7, 0.7-0.85, 0.85-1.0)
5. Randomly sample `N_PER_BIN` from each bin → final 100 molecules
6. Save final seeds JSON (drop-in replacement for `seeds_for_methods.json`)

**Outputs** (`data/generation_stratified/`):
- `chembl_sampled_stratified.csv` — final 100 SMILES
- `chembl_features_stratified.npz` — features for the final 100
- `seeds_for_methods_stratified.json` — seeds with balanced quality bins

> **Kernel:** `prexsyn_env` (Python 3.11)

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import requests
from tqdm.notebook import tqdm
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, Descriptors

from src.evaluation.scoring_v2 import make_spec, tanimoto_to_spec
from src.utils.featurize_chembl import featurize

# ── Paths ─────────────────────────────────────────────────────────────────────
CHEMBL_FULL_CSV = ROOT / 'data' / 'chembl_full.csv'

# Change OUT_DIR here to create a new sample without overwriting the original.
# original : data/generation_stratified        (N_PER_BIN=25, 100 seeds)
# large    : data/generation_stratified_large  (N_PER_BIN=50, 200 seeds)
OUT_DIR = ROOT / 'data' / 'generation_stratified_large'
OUT_DIR.mkdir(parents=True, exist_ok=True)

OVERSAMPLE_CSV  = OUT_DIR / 'chembl_oversampled.csv'
OVERSAMPLE_NPZ  = OUT_DIR / 'chembl_oversampled_features.npz'
OVERSAMPLE_JSON = OUT_DIR / 'prexsyn_oversample_seeds.json'
SAMPLED_CSV     = OUT_DIR / 'chembl_sampled_stratified.csv'
FEATURES_NPZ    = OUT_DIR / 'chembl_features_stratified.npz'
SEEDS_JSON      = OUT_DIR / 'seeds_for_methods_stratified.json'

# ── Parameters ────────────────────────────────────────────────────────────────
N_OVERSAMPLE      = 800  # molecules to run PrexSyn on (>= 4 * N_PER_BIN)
N_PER_BIN         = 50    # molecules to keep per quality bin -> 200 total
N_PREXSYN_SAMPLES = 256
RANDOM_SEED       = 42
PREXSYN_URL       = 'http://localhost:8011/sample'

# Set to False to skip Ro5 and only validate SMILES.
APPLY_RO5 = True

QUALITY_BINS = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']

print(f'ChEMBL full  : {CHEMBL_FULL_CSV.exists()}  ({CHEMBL_FULL_CSV.name})')
print(f'Output dir   : {OUT_DIR}')
print(f'Ro5 filter   : {APPLY_RO5}')
print(f'Oversample N : {N_OVERSAMPLE}  ->  keep {N_PER_BIN}/bin = {N_PER_BIN*4} final')
print()
print('Cache status:')
for label, path in [
    ('oversample CSV ', OVERSAMPLE_CSV),
    ('oversample NPZ ', OVERSAMPLE_NPZ),
    ('oversample seeds', OVERSAMPLE_JSON),
    ('final seeds     ', SEEDS_JSON),
]:
    print(f'  {label} : {"[cached]" if path.exists() else "[missing]"}  {path.name}')

ChEMBL full  : True  (chembl_full.csv)
Output dir   : D:\AI4DD Project\CSC2541-Project\data\generation_stratified_large
Ro5 filter   : True
Oversample N : 1600  ->  keep 50/bin = 200 final

Cache status:
  oversample CSV  : [missing]  chembl_oversampled.csv
  oversample NPZ  : [missing]  chembl_oversampled_features.npz
  oversample seeds : [missing]  prexsyn_oversample_seeds.json
  final seeds      : [missing]  seeds_for_methods_stratified.json


---
## Stage 1 — Sample `N_OVERSAMPLE` molecules from ChEMBL

Randomly draw `N_OVERSAMPLE` molecules first, then optionally apply Ro5.
This avoids scanning all 1.79 M rows before sampling.

In [2]:
if OVERSAMPLE_CSV.exists():
    df_over = pd.read_csv(OVERSAMPLE_CSV)
    print(f'[cached] {len(df_over)} molecules <- {OVERSAMPLE_CSV.name}')
else:
    print('Loading ChEMBL full CSV...')
    df_full    = pd.read_csv(CHEMBL_FULL_CSV)
    smiles_col = df_full.columns[0]
    all_smiles = df_full[smiles_col].dropna().tolist()
    print(f'Total raw: {len(all_smiles):,}')

    # Step 1: random sample first
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(all_smiles), size=min(N_OVERSAMPLE, len(all_smiles)), replace=False)
    sampled = [all_smiles[i] for i in sorted(idx)]

    # Step 2: validate SMILES + optional Ro5 filter
    valid = []
    for smi in tqdm(sampled, desc='Ro5 filter' if APPLY_RO5 else 'Validating SMILES'):
        smi = str(smi).strip()
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        if APPLY_RO5:
            if '.' in smi:
                continue
            if (Descriptors.MolWt(mol)           > 500 or
                Descriptors.MolLogP(mol)         > 5   or
                rdMolDescriptors.CalcNumHBD(mol) > 5   or
                rdMolDescriptors.CalcNumHBA(mol) > 10):
                continue
        valid.append(Chem.MolToSmiles(mol))

    valid = list(dict.fromkeys(valid))
    print(f'After {"Ro5" if APPLY_RO5 else "validity"} filter: {len(valid)} / {len(sampled)}')

    df_over = pd.DataFrame({'SMILES': valid})
    df_over.to_csv(OVERSAMPLE_CSV, index=False)
    print(f'Saved -> {OVERSAMPLE_CSV.name}')


Loading ChEMBL full CSV...
Total raw: 1,794,749


Ro5 filter:   0%|          | 0/1600 [00:00<?, ?it/s]

After Ro5 filter: 1121 / 1600
Saved -> chembl_oversampled.csv


---
## Stage 2 — Featurize the oversampled pool

In [3]:
if OVERSAMPLE_NPZ.exists():
    print(f'[cached] <- {OVERSAMPLE_NPZ.name}')
else:
    featurize(input=OVERSAMPLE_CSV, output=OVERSAMPLE_NPZ)

_data        = np.load(OVERSAMPLE_NPZ, allow_pickle=True)
smiles_arr   = _data['smiles']
ecfp4_arr    = _data['ecfp4']
fcfp4_arr    = _data['fcfp4']
rdkit_vals   = _data['rdkit_desc_values']
rdkit_names  = _data['rdkit_desc_names'].tolist()
brics_fps    = _data['brics_fps']
brics_exists = _data['brics_exists']

feat = {
    smi: {
        'ecfp4':  ecfp4_arr[i],
        'fcfp4':  fcfp4_arr[i],
        'rdkit':  rdkit_vals[i],
        'brics':  brics_fps[i],
        'bric_e': brics_exists[i],
    }
    for i, smi in enumerate(smiles_arr)
}
print(f'Features ready for {len(smiles_arr)} molecules')


Loaded 1121 molecules from D:\AI4DD Project\CSC2541-Project\data\generation_stratified_large\chembl_oversampled.csv


Featurizing: 100%|██████████| 1121/1121 [00:04<00:00, 255.83it/s]


Featurized 1121 molecules.
Saved features to: D:\AI4DD Project\CSC2541-Project\data\generation_stratified_large\chembl_oversampled_features.npz
Features ready for 1121 molecules


---
## Stage 3 — Run PrexSyn on the full oversampled pool

Records actual `baseline_quality` (Tanimoto of best candidate to spec) for every molecule.

In [4]:
def _quality_bin(bq: float) -> str:
    if   bq < 0.50: return '<0.5'
    elif bq < 0.70: return '0.5-0.7'
    elif bq < 0.85: return '0.7-0.85'
    else:           return '0.85-1.0'


if OVERSAMPLE_JSON.exists():
    with open(OVERSAMPLE_JSON) as f:
        all_seeds = json.load(f)
    print(f'[cached] {len(all_seeds)} PrexSyn results <- {OVERSAMPLE_JSON.name}')
else:
    all_seeds  = []
    api_errors = []

    for smi in tqdm(smiles_arr.tolist(), desc='PrexSyn (oversample)'):
        f = feat[smi]
        payload = {
            'ecfp4':             f['ecfp4'].tolist(),
            'fcfp4':             f['fcfp4'].tolist(),
            'rdkit_desc_values': f['rdkit'].tolist(),
            'rdkit_desc_names':  rdkit_names,
            'brics_fps':         f['brics'].tolist(),
            'brics_exists':      f['bric_e'].tolist(),
            'source_smiles':     smi,
            'num_samples':       N_PREXSYN_SAMPLES,
        }
        try:
            resp       = requests.post(PREXSYN_URL, json=payload, timeout=300)
            resp.raise_for_status()
            candidates = resp.json().get('generated_smiles', [])
        except Exception as e:
            api_errors.append(f'{smi[:40]}: {e}')
            candidates = []

        spec = make_spec(smi)
        best_smi, best_t = smi, 0.0
        if spec and candidates:
            for c in candidates:
                if '.' in c:
                    continue
                mol = Chem.MolFromSmiles(c)
                if mol:
                    t = tanimoto_to_spec(mol, spec)
                    if t > best_t:
                        best_t, best_smi = t, c

        all_seeds.append({
            'spec_smiles':      smi,
            'seed_smiles':      best_smi,
            'baseline_quality': round(best_t, 4),
            'quality_bin':      _quality_bin(best_t),
            'methods':          {},
        })

    with open(OVERSAMPLE_JSON, 'w') as f:
        json.dump(all_seeds, f, indent=2)
    print(f'Saved {len(all_seeds)} seeds -> {OVERSAMPLE_JSON.name}')
    if api_errors:
        print(f'  API errors: {len(api_errors)}')

from collections import Counter
pool_bins = Counter(s['quality_bin'] for s in all_seeds)
print('\nPool quality-bin distribution:')
for b in QUALITY_BINS:
    print(f'  {b:<10} : {pool_bins[b]:>4}  (need {N_PER_BIN})')


PrexSyn (oversample):   0%|          | 0/1121 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## Stage 4 — Stratified downsample to `N_PER_BIN` per quality bin

In [ ]:
if SEEDS_JSON.exists():
    with open(SEEDS_JSON) as f:
        seeds = json.load(f)
    print(f'[cached] {len(seeds)} final seeds <- {SEEDS_JSON.name}')
else:
    rng = np.random.default_rng(RANDOM_SEED)

    bins_dict = {b: [] for b in QUALITY_BINS}
    for s in all_seeds:
        bins_dict[s['quality_bin']].append(s)

    seeds = []
    print('Stratified downsample:')
    for b in QUALITY_BINS:
        pool = bins_dict[b]
        n    = min(N_PER_BIN, len(pool))
        if n < N_PER_BIN:
            print(f'  WARNING: {b} has only {len(pool)} molecules (need {N_PER_BIN})')
        chosen = rng.choice(len(pool), size=n, replace=False)
        seeds.extend([pool[i] for i in chosen])
        print(f'  {b:<10} : kept {n} / {len(pool)}')

    with open(SEEDS_JSON, 'w') as f:
        json.dump(seeds, f, indent=2)
    print(f'\nSaved {len(seeds)} final seeds -> {SEEDS_JSON.name}')

    pd.DataFrame({'SMILES': [s['spec_smiles'] for s in seeds]}).to_csv(SAMPLED_CSV, index=False)

    idx_map  = {smi: i for i, smi in enumerate(smiles_arr.tolist())}
    keep_idx = [idx_map[s['spec_smiles']] for s in seeds if s['spec_smiles'] in idx_map]
    np.savez_compressed(
        FEATURES_NPZ,
        smiles=smiles_arr[keep_idx],
        ecfp4=ecfp4_arr[keep_idx],
        fcfp4=fcfp4_arr[keep_idx],
        rdkit_desc_values=rdkit_vals[keep_idx],
        rdkit_desc_names=_data['rdkit_desc_names'],
        brics_fps=brics_fps[keep_idx],
        brics_exists=brics_exists[keep_idx],
    )
    print(f'Saved features for {len(keep_idx)} molecules -> {FEATURES_NPZ.name}')


---
## Stage 5 — Verification

In [ ]:
import matplotlib.pyplot as plt

seeds_df = pd.DataFrame([
    {'baseline_quality': s['baseline_quality'], 'quality_bin': s['quality_bin']}
    for s in seeds
])

ORIG_BINS = {'<0.5': 33, '0.5-0.7': 31, '0.7-0.85': 11, '0.85-1.0': 25}
final_counts = seeds_df['quality_bin'].value_counts().reindex(QUALITY_BINS, fill_value=0)

print(f"{'Bin':<12}  {'Original':>10}  {'Stratified':>12}")
for b in QUALITY_BINS:
    print(f"{b:<12}  {ORIG_BINS[b]:>10}  {final_counts[b]:>12}")
print(f"\nbaseline_quality: mean={seeds_df['baseline_quality'].mean():.3f}"
      f"  std={seeds_df['baseline_quality'].std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

x, w = range(4), 0.35
axes[0].bar([i - w/2 for i in x], [ORIG_BINS[b]    for b in QUALITY_BINS], width=w,
            label='Original (n=100)', color='steelblue')
axes[0].bar([i + w/2 for i in x], [final_counts[b] for b in QUALITY_BINS], width=w,
            label='Stratified (n=100)', color='coral')
axes[0].axhline(N_PER_BIN, color='gray', linestyle='--', linewidth=0.8, label=f'Target n={N_PER_BIN}')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(QUALITY_BINS)
axes[0].set_ylabel('Count')
axes[0].set_title('Quality bin distribution')
axes[0].legend(fontsize=8)

axes[1].hist(seeds_df['baseline_quality'], bins=20, edgecolor='white', color='coral')
for v in (0.5, 0.7, 0.85):
    axes[1].axvline(v, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Baseline quality')
axes[1].set_ylabel('Count')
axes[1].set_title('PrexSyn quality distribution — stratified sample')

plt.tight_layout()
plt.savefig(OUT_DIR / 'stratified_quality_dist.png', dpi=150)
plt.show()


---
## Next Steps

Point `data_generation.ipynb` at the new outputs:

```python
# Cell 1 of data_generation.ipynb
GEN_DIR    = ROOT / 'data' / 'generation_stratified'
SEEDS_JSON = GEN_DIR / 'seeds_for_methods_stratified.json'
```

Set all `USE_CACHED_*` flags to `False` and re-run the method generation cells.